[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/32_nlp/01_text_preprocessing_and_tokenization.ipynb)

# Bài 01 — Tiền xử lý văn bản và tokenization

**🎯 Mục tiêu**
- Các bước biến văn bản thô thành chuỗi token; những cạm bẫy riêng của **tiếng Việt**: Unicode NFC/NFD, kiểu bỏ dấu cũ/mới, gõ không dấu.
- Tách **âm tiết** vs tách **từ** (từ ghép nhiều âm tiết); cài **tách từ cực đại** (maximum matching) và thấy giới hạn của nó.
- Stopword, chuẩn hoá: khi nào giúp, khi nào hại.
- **Subword tokenization**: cài **BPE** from scratch; đánh đổi kích thước từ vựng ↔ độ dài chuỗi ↔ từ chưa gặp (OOV).
- Định luật **Zipf**: vì sao từ vựng luôn có "đuôi dài".

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/32_nlp

In [ ]:
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.datasets import NEWS_LEXICON, make_news_vi
from ml_course.visualization import set_style

set_style()
news = make_news_vi(3000, seed=0)
for t in news.sample(4, random_state=1).itertuples():
    print(f"[{t.topic:9s}] {t.text}")

Ngữ liệu của chương: 3 000 tin ngắn **sinh tự động** thuộc 6 chủ đề (thể thao, kinh tế, công nghệ, sức khỏe, giáo dục, du lịch) — câu đôi khi không tự nhiên, nhưng có các hiện tượng thật: từ đồng nghĩa ("smartphone" / "điện thoại thông minh"), từ dùng chung giữa chủ đề ("thị trường"), câu lạc đề ("Trong khi đó, ..."), và nhãn nhiễu.

## 1. Unicode: cùng chữ, khác byte

Một ký tự có dấu tiếng Việt có thể mã hoá theo hai cách:
- **NFC** (dựng sẵn): "ộ" là **một** code point (U+1ED9).
- **NFD** (tổ hợp): "o" + dấu mũ + dấu nặng — **ba** code point.

Văn bản copy từ các nguồn khác nhau (Word, web, macOS) trộn lẫn hai dạng → chuỗi **nhìn giống hệt** nhưng so sánh `==` ra `False`, từ điển đếm thành hai từ khác nhau. Ngoài ra còn hai **kiểu bỏ dấu**: "hoà" (cũ) và "hòa" (mới) — khác cả về chữ.

In [ ]:
a = unicodedata.normalize("NFC", "Việt Nam")
b = unicodedata.normalize("NFD", "Việt Nam")
print(f"nhìn: {a!r} vs {b!r} | bằng nhau? {a == b} | độ dài {len(a)} vs {len(b)}")
print("code point của 'ệ' ở NFD:", [f"U+{ord(c):04X} {unicodedata.name(c)}" for c in unicodedata.normalize("NFD", "ệ")])
print("sau khi chuẩn hoá NFC cả hai:", unicodedata.normalize("NFC", b) == a)
print("kiểu bỏ dấu cũ/mới:", "hoà" == "hòa", "| 'thuỷ' vs 'thủy':", "thuỷ" == "thủy")

**Quy tắc**: chuẩn hoá NFC (`unicodedata.normalize("NFC", s)`) **ngay khi đọc dữ liệu**, và thống nhất một kiểu bỏ dấu (bảng chuyển "oà → òa", "uỷ → ủy"...). Hai dòng code này tránh được nhiều lỗi khó tìm.

## 2. Gõ không dấu và bỏ dấu

Người dùng thường gõ không dấu ("doi tra hang"). Bỏ dấu để **so khớp** là hữu ích (bài 04 cho thấy tìm kiếm FAQ cải thiện rõ), nhưng mất thông tin: nhiều từ khác nghĩa trùng nhau sau khi bỏ dấu.

In [ ]:
def strip_accents(s):
    s = unicodedata.normalize("NFD", s).replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in s if unicodedata.category(c) != "Mn")   # Mn: dấu kết hợp


print([strip_accents(w) for w in ["ma", "má", "mà", "mả", "mã", "mạ", "đường"]])

syllables_all = Counter(s for t in news.text for s in re.findall(r"\w+", t.lower()))
groups = {}
for s in syllables_all:
    groups.setdefault(strip_accents(s), []).append(s)
collisions = {k: v for k, v in groups.items() if len(v) > 1}
print(f"{len(syllables_all)} âm tiết khác nhau → {len(groups)} sau khi bỏ dấu | {len(collisions)} nhóm bị trùng, ví dụ:",
      list(collisions.items())[:5])

## 3. Làm sạch cơ bản

Tuỳ bài toán: chữ thường, bỏ/thay URL, email, số, ký tự đặc biệt, emoji, khoảng trắng thừa. Không có "bộ làm sạch chuẩn" — mỗi bước là một **giả định** về cái gì không mang thông tin. Với phân tích cảm xúc, "!!!" và emoji có thể rất quan trọng; với phân loại chủ đề thì không.

In [ ]:
def clean(text):
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"https?://\S+", " <url> ", text)
    text = re.sub(r"\S+@\S+\.\S+", " <email> ", text)
    text = re.sub(r"\d+([.,/]\d+)*", " <số> ", text)
    text = re.sub(r"[^\w\s<>]", " ", text)                             # bỏ dấu câu (giữ chữ có dấu tiếng Việt: \w)
    return re.sub(r"\s+", " ", text).strip()


print(clean("Giảm 50% cho đơn 300.000đ!!! Xem tại https://shop.vn/sale hoặc gửi mail cho cskh@shop.vn 😊"))

## 4. Âm tiết hay từ? Tách từ tiếng Việt

Tiếng Việt viết cách nhau theo **âm tiết**, nhưng **từ** có thể gồm nhiều âm tiết: "huấn luyện viên", "điện thoại thông minh". Tách theo khoảng trắng cho các âm tiết — "điện", "thoại" mất nghĩa riêng lẻ. **Tách từ** (word segmentation) ghép chúng: `huấn_luyện_viên`.

Cách đơn giản nhất: **khớp cực đại tiến** (forward maximum matching) với một từ điển — tại mỗi vị trí, lấy từ **dài nhất** có trong từ điển. Từ điển ở đây là `NEWS_LEXICON` (các từ nhiều âm tiết của ngữ liệu).

In [ ]:
LEXICON = {tuple(w.lower().split()) for w in NEWS_LEXICON}
MAX_LEN = max(map(len, LEXICON))


def segment(text, lexicon=LEXICON):
    syl = re.findall(r"\w[\w\-]*", unicodedata.normalize("NFC", text).lower())
    tokens, i = [], 0
    while i < len(syl):
        for L in range(min(MAX_LEN, len(syl) - i), 0, -1):            # thử từ dài nhất trước
            if L == 1 or tuple(syl[i:i + L]) in lexicon:
                tokens.append("_".join(syl[i:i + L]))
                i += L
                break
    return tokens


sample = news.text[0]
print(sample)
print("âm tiết:", re.findall(r"\w[\w\-]*", sample.lower()))
print("từ     :", segment(sample))

Khớp cực đại nhanh và đơn giản nhưng **tham lam** — không xét ngữ cảnh. Ví dụ kinh điển: "Học sinh học sinh học" (= học sinh / học / sinh học).

In [ ]:
toy_lexicon = {("học", "sinh"), ("sinh", "học")}
print("khớp cực đại:", segment("Học sinh học sinh học", toy_lexicon))
print("đúng là     : ['học_sinh', 'học', 'sinh_học']")

Khớp tham lam từ trái sang phải chọn "học_sinh" lần thứ hai → sai. Công cụ thực tế (VnCoreNLP, underthesea, pyvi) dùng mô hình thống kê/học máy (CRF, mạng nơ-ron) huấn luyện trên ngữ liệu đã tách từ, đạt độ chính xác ~97–98%. Mô hình Transformer hiện đại thường **bỏ qua** tách từ và dùng **subword** (mục 6) — riêng PhoBERT yêu cầu đầu vào đã tách từ.

## 5. Stopword và chuẩn hoá: cẩn thận

**Stopword**: từ rất phổ biến, ít mang nội dung ("của", "là", "và", "thì"...). Bỏ chúng giảm kích thước từ vựng cho các mô hình đếm từ — nhưng danh sách stopword phổ biến thường chứa "**không**", "chưa", "rất" — bỏ đi thì "không hài lòng" thành "hài lòng". Với mô hình hiện đại (TF-IDF đã tự giảm trọng số từ phổ biến, embedding, Transformer), thường **không cần** bỏ stopword.

In [ ]:
stopwords = {"của", "là", "và", "thì", "những", "các", "không", "rất", "có"}   # danh sách "ngây thơ"
review = "sản phẩm không tốt và giao hàng không nhanh"
print("trước:", review)
print("sau  :", " ".join(w for w in review.split() if w not in stopwords), "← nghĩa bị đảo ngược")

## 6. Subword tokenization: Byte Pair Encoding (BPE)

Từ vựng theo từ gặp hai vấn đề: **rất lớn** (hàng trăm nghìn từ, cộng tên riêng, lỗi chính tả) và **OOV** — từ chưa gặp khi train bị thay bằng `<unk>`. Theo ký tự thì không có OOV nhưng chuỗi rất dài. **BPE** (Sennrich et al., 2016) đi giữa:

1. Bắt đầu với từ vựng = các ký tự; mỗi từ là chuỗi ký tự + ký hiệu kết thúc `</w>`.
2. Lặp: đếm mọi **cặp ký hiệu kề nhau** trong ngữ liệu (theo tần suất từ), **gộp** cặp phổ biến nhất thành ký hiệu mới, ghi lại phép gộp.
3. Dừng khi đủ số phép gộp → từ vựng = ký tự + các ký hiệu đã gộp.

Khi tokenize từ mới: áp dụng các phép gộp **theo đúng thứ tự đã học**. Từ phổ biến thành 1 token; từ hiếm/lạ vỡ thành vài mảnh quen thuộc — không bao giờ OOV (trừ ký tự chưa từng thấy; byte-level BPE của GPT xử lý cả điều đó).

In [ ]:
def train_bpe(word_counts, n_merges):
    vocab = {tuple(w) + ("</w>",): c for w, c in word_counts.items()}
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for symbols, c in vocab.items():
            for a, b in zip(symbols, symbols[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        new_vocab = {}
        for symbols, c in vocab.items():                               # gộp mọi lần xuất hiện của cặp tốt nhất
            out, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best:
                    out.append(symbols[i] + symbols[i + 1])
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            new_vocab[tuple(out)] = c
        vocab = new_vocab
    return merges


def bpe_tokenize(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:                                                # áp dụng theo đúng thứ tự đã học
        i, out = 0, []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        symbols = out
    return symbols


train_text, test_text = news.text[:2500], news.text[2500:]
word_counts = Counter(w for t in train_text for w in re.findall(r"\w+", t.lower()))
merges = train_bpe(word_counts, 300)
print("10 phép gộp đầu tiên:", ["".join(m) for m in merges[:10]])
for w in ["thông", "smartphone", "smarphone", "điệnthoại", "metaverse"]:
    print(f"{w:12s} → {bpe_tokenize(w, merges)}")

Các phép gộp đầu tiên là những cặp ký tự phổ biến nhất của tiếng Việt ("ng", "nh", "th", "ch"...). Âm tiết rất phổ biến ("thông") đã thành 1 token; từ ít gặp hơn ("smartphone" — chỉ xuất hiện trong một chủ đề con) chưa được gộp hết sau 300 phép gộp nên vỡ thành vài mảnh; lỗi chính tả ("smarphone"), viết dính ("điệnthoại") hay từ mới ("metaverse") vẫn biểu diễn được bằng các mảnh quen thuộc thay vì `<unk>`. Càng nhiều phép gộp, càng nhiều từ thành một token.

### Đánh đổi: kích thước từ vựng ↔ độ dài chuỗi ↔ OOV

So sánh trên phần test (500 tin chưa dùng khi xây từ vựng), có thêm 5% âm tiết bị **gõ sai** (đổi một ký tự) để mô phỏng dữ liệu thật:

In [ ]:
rng = np.random.default_rng(0)
letters = "abcdeghiklmnopqrstuvxy"


def add_typos(text, rate=0.05):
    words = re.findall(r"\w+", text.lower())
    out = []
    for w in words:
        if len(w) > 2 and rng.random() < rate:
            i = rng.integers(len(w))
            w = w[:i] + letters[rng.integers(len(letters))] + w[i + 1:]
        out.append(w)
    return out


test_words = [w for t in test_text for w in add_typos(t)]
syllable_vocab = set(word_counts)
char_vocab = set("".join(word_counts))
rows = []
for name, n_merges in [("ký tự", 0), ("BPE 100 phép gộp", 100), ("BPE 300 phép gộp", 300)]:
    m = merges[:n_merges]
    cache = {}
    n_tokens = sum(len(cache.setdefault(w, bpe_tokenize(w, m))) for w in test_words)
    n_unk = sum(any(s not in char_vocab and s != "</w>" and len(s) == 1 for s in cache[w]) for w in test_words)
    rows.append((name, len(char_vocab) + 1 + n_merges, n_tokens / len(test_text), n_unk / len(test_words)))
oov = np.mean([w not in syllable_vocab for w in test_words])
rows.append(("âm tiết (từ vựng train)", len(syllable_vocab) + 1, len(test_words) / len(test_text), oov))
print(pd.DataFrame(rows, columns=["cách tách", "kích thước từ vựng", "token / tin", "tỉ lệ token không biểu diễn được"]).round(3).to_string(index=False))

- **Ký tự**: từ vựng nhỏ nhất, không OOV, nhưng chuỗi dài nhất (mô hình phải "học chính tả" và tốn tính toán — attention $O(n^2)$, chương 31).
- **Âm tiết/từ**: chuỗi ngắn nhất nhưng mọi âm tiết gõ sai thành OOV (`<unk>`) — thông tin mất hẳn.
- **BPE**: kích thước từ vựng là **siêu tham số**; tăng số phép gộp → chuỗi ngắn dần về phía từ, vẫn không OOV. LLM thực tế dùng 30 000–200 000 token (BPE, WordPiece, Unigram/SentencePiece; byte-level để bao phủ mọi ký tự).

Lưu ý thêm: tokenizer huấn luyện chủ yếu trên tiếng Anh thường cắt tiếng Việt thành **nhiều token hơn** cho cùng nội dung → ngữ cảnh "chứa" ít nội dung hơn và chi phí API cao hơn — một lý do các mô hình cho tiếng Việt huấn luyện tokenizer riêng.

## 7. Định luật Zipf

Tần suất từ giảm theo hạng gần như **luỹ thừa**: từ hạng $r$ có tần suất $\propto 1/r^{s}$. Hệ quả: vài từ chiếm phần lớn văn bản, còn rất nhiều từ hiếm — luôn có OOV, và các mô hình đếm từ cần làm mịn (smoothing). **Lưu ý**: ngữ liệu tổng hợp ở đây sinh từ một danh sách từ cố định nên chỉ có vài trăm âm tiết và **không có đuôi dài** (không âm tiết nào hiếm) — khác hẳn văn bản thật, nơi khoảng một nửa số từ khác nhau chỉ xuất hiện một lần và đường log–log gần thẳng suốt nhiều bậc độ lớn. Đây là một giới hạn cần nhớ khi đọc các kết quả của chương: dữ liệu thật "khó" hơn.

In [ ]:
freqs = np.array(sorted(syllables_all.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
slope = np.polyfit(np.log(ranks[:200]), np.log(freqs[:200]), 1)[0]
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ranks, freqs, ".", ms=3)
ax.set_xlabel("hạng (log)")
ax.set_ylabel("tần suất (log)")
ax.set_title(f"Zipf trên ngữ liệu tin tức: độ dốc ≈ {slope:.2f} (200 âm tiết đầu)")
plt.tight_layout()
plt.show()
top10 = freqs[:10].sum() / freqs.sum()
print(f"{len(freqs)} âm tiết khác nhau | 10 âm tiết phổ biến nhất chiếm {top10:.0%} tổng số | âm tiết hiếm nhất vẫn xuất hiện {freqs[-1]} lần")

## 🧠 Kiểm tra nhanh

1. Vì sao hai chuỗi "Việt Nam" nhìn giống nhau lại có thể không bằng nhau? Cách khắc phục?
2. Bỏ dấu tiếng Việt có lợi và có hại gì?
3. Tách từ khác tách âm tiết thế nào? Vì sao khớp cực đại có thể sai?
4. Khi nào **không** nên bỏ stopword?
5. Mô tả thuật toán BPE. Vì sao BPE không có OOV? Số phép gộp ảnh hưởng gì?

## 📌 Tóm tắt

- Chuẩn hoá Unicode NFC và kiểu bỏ dấu ngay từ đầu; bỏ dấu giúp so khớp với người dùng gõ không dấu nhưng gây trùng nghĩa.
- Tiếng Việt cách theo âm tiết; tách từ (khớp cực đại, CRF, mô hình nơ-ron) ghép từ nhiều âm tiết; mô hình hiện đại thường dùng subword.
- Làm sạch và stopword là giả định — bỏ "không" có thể đảo nghĩa.
- BPE: gộp dần cặp ký hiệu phổ biến; đánh đổi từ vựng ↔ độ dài chuỗi; không OOV.
- Zipf: đuôi dài từ hiếm là bản chất của ngôn ngữ.

## 📚 Tài liệu tham khảo

- Sennrich, Haddow & Birch (2016) — *Neural Machine Translation of Rare Words with Subword Units* (BPE)
- Kudo & Richardson (2018) — *SentencePiece*
- Vu et al. (2018) — *VnCoreNLP: A Vietnamese Natural Language Processing Toolkit*
- Jurafsky & Martin — *Speech and Language Processing*, chương 2 (Regular Expressions, Tokenization)
- [Hugging Face — Tokenizers summary](https://huggingface.co/docs/transformers/tokenizer_summary)